# 12 pamoka – Pokalbių istorijos mažinimas su agente Scratchpad

Ši užrašų knygutė demonstruoja, kaip valdyti kontekstą ilguose pokalbiuose naudojant Microsoft Agent Framework. Pokalbiams plečiantis, didėja žetonų skaičius — galiausiai viršijamas modelio konteksto langas. Tai sprendžiame naudodami **konteksto santraukos modelį** ir **agento Scratchpad** nuolatinei atminčiai.

## Ko išmoksite:
1. **Kodėl svarbus konteksto valdymas**: suprasti žetonų ribas ir konteksto langus
2. **Konteksto suvokiantys agentai**: kurti agentus, kurie patys valdo savo pokalbio kontekstą
3. **Konteksto santraukos modelis**: naudoti įrankius pokalbių istorijai sutrumpinti
4. **Agento Scratchpad**: nuolatinė atmintis, išgyvenanti konteksto mažinimą

## Reikalavimai:
- Azure OpenAI nustatymas su aplinkos kintamaisiais
- Pagrindinių agentų sampratų iš ankstesnių pamokų supratimas


## Diegimas


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Kodėl svarbu valdyti kontekstą

Kiekvienas LLM turi ribotą **konteksto langą** – maksimalų žodžių skaičių, kurį jis gali apdoroti viename užklausoje. Pokalbiui vykstant keliais etapais:

- **Žodžių skaičius auga tiesiškai** su kiekvienu naudotojo pranešimu ir asistento atsakymu.
- **Prašymo žodžiai sudaro didžiąją kainos dalį**, nes visa istorija siunčiama kiekviename žingsnyje.
- Galiausiai pokalbis **viršija konteksto langą** ir modelis arba sutrumpina, arba meta klaidą.

### Konteksto valdymo strategijos

| Strategija | Kaip veikia | Kompromisas |
|---|---|---|
| **Sutrumpinimas** | Išmetami seniausi pranešimai | Prarandamas ankstyvas kontekstas |
| **Santrauka** | Senesni pranešimai sutrumpinami į santrauką | Dalinė informacija prarandama, bet pagrindinės mintys išlieka |
| **Rašymo bloknotas / Išorinė atmintis** | Svarbios faktos saugomos už pokalbio ribų | Reikalauja įrankių iškvietimų, bet išlieka nepaisant sutrumpinimo |

Šiame užrašų knygelėje deriname **santrauką** su **rašymo bloknoto įrankiu**, kad agentas galėtų išlaikyti tęstinumą net kai pokalbio istorija sutrumpinama.


## Kontextą suvokiančio agento kūrimas


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Ilgos Pokalbio Simuliacija

Pažvelkime į daugiaprasmį pokalbį, kad pamatytume, kaip kaupiasi kontekstas. Agentas turėtų išlaikyti svarbias detales (pageidavimus, biudžetą, kelionės datas) per visus pokalbio etapus ir parodyti tęstinumą.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Atkreipkite dėmesį, kaip agentas išlaiko ankstesnių pokalbių kontekstą – jis žino apie Japoniją, suši, šventoves, fotografiją, 3000 $ biudžetą, keliones vienam ir balandžio laikotarpį. Trumpame pokalbyje tai veikia gerai, tačiau ilgėjant pokalbiui visos istorijos perdavimas tampa brangus.

Tęskime pokalbį su daugiau turimų kalbų, kad pamatytume, kaip kaupiasi kontekstas:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Konteksto santraukos šablonas

Kai pokalbis auga, galime naudoti **santraukos įrankį**, kad suspaustume sukauptą kontekstą į kompaktišką formatą. Agentas iškviečia šį įrankį, kad įrašytų pagrindines nuostatas, taigi net jei senesni pranešimai bus pašalinti, svarbi informacija išliks.

Šis šablonas yra pažangesnio istorijos mažinimo pagrindas:
1. Agentas identifikuoja pagrindinius faktus iš pokalbio
2. Jis iškviečia santraukos įrankį, kad juos įrašytų
3. Senesni pranešimai gali būti saugiai pašalinti, nes santrauka apima svarbiausią informaciją

Žemiau apibrėžiame `summarize_preferences` įrankį, kurį agentas gali iškviesti, norėdamas įrašyti kompaktišką to, ką jis sužinojo, santrauką.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Santrauka

Šiame pamokoje sužinojote, kaip valdyti kontekstą ilgai trunkančiuose agentų pokalbiuose naudojant Microsoft Agent Framework:

### Pagrindinės sąvokos
- **Konteksto langai yra riboti** — kiekvienas pokalbio istorijos žetonas kainuoja pinigus ir skaičiuojamas prie ribos.
- **Santraukos įrankiai** leidžia agentui sutraukti sukauptą kontekstą į kompaktiškas santraukas, sumažinant žetonų naudojimą išlaikant esminę informaciją.
- **Agentų užrašų knygelės** suteikia nuolatinę išorinę atmintį, kuri išlieka nepriklausomai nuo pokalbio sutrumpinimo.

### Ką sukūrėte
- **Konteksto suvokiantis agentas**, kuris palaiko tęstinumą kelių kartų pokalbiuose
- **Santraukos įrankis** (`summarize_preferences`), įrašantis svarbiausius vartotojo duomenis kompaktiškame formate
- **Kelių kartų pokalbis**, demonstravęs konteksto išlaikymą ir pokyčių valdymą

### Realūs taikymai
- **Klientų aptarnavimo botai**: prisimena vartotojo pageidavimus ilgų palaikymo sesijų metu
- **Asmeniniai asistentai**: seka vykstančius projektus be būtinybės vėl aiškinti kontekstą
- **Mokomieji mokytojai**: palaiko mokinių pažangą per daugelį sąveikų

### Kiti žingsniai
- Įgyvendinti pilną užrašų knygelės įrankį su failų pagrindu veikiančia saugykla
- Pridėti automatinį istorijos sutrumpinimą po santraukos sudarymo
- Integruoti su vektorinėmis duomenų bazėmis semantiniam atminties paieškai
- Kurti agentus, galinčius tęsti pokalbius po kelių dienų išlaikant visą kontekstą


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės atsisakymas**:
Šis dokumentas buvo išverstas naudojant AI vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, atkreipkite dėmesį, kad automatiniai vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba turėtų būti laikomas autoritetingu šaltiniu. Svarbiai informacijai rekomenduojame profesionalų žmogišką vertimą. Mes neprisiimame atsakomybės už bet kokius nesusipratimus ar neteisingus interpretavimus, kylančius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
